# QuantLab, notebook edition

Same engine as the Streamlit app, no UI. Runs in Google Colab, Microsoft Fabric or a local Jupyter.

The flow is the same as the tabs: data → features → labels → walk-forward → backtest → "is this real?" → log → compare.
Every headline number is computed on the concatenated out-of-sample series only.

**Colab:** run the setup cell first, it clones the repo and installs what is missing. Runtime → Run all works after that.
**Local:** open this from the repo root with the venv active and the setup cell is a no-op.

In [ ]:
# --- Setup: find or fetch the repo, install what is missing, make `quantlab` importable ---------------
import importlib, os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/AntonAlin/Quant-lab.git"
BRANCH = "claude/laughing-fermat-1jw558"

def find_repo_root() -> Path | None:
    for cand in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Quant-lab", Path("/content/Quant-lab")]:
        if (cand / "quantlab" / "app.py").exists():
            return cand
    return None

root = find_repo_root()
if root is None:
    # Not inside the repo (hello, Colab). Clone it next to the notebook.
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL], check=True)
    root = Path.cwd() / "Quant-lab"
sys.path.insert(0, str(root))
os.chdir(root)

# Install pinned requirements, but do not re-download 2 GB of torch if one is already here (Colab ships it).
reqs = [l.strip() for l in (root / "requirements.txt").read_text().splitlines() if l.strip() and not l.startswith("#")]
have_torch = importlib.util.find_spec("torch") is not None
missing = []
for r in reqs:
    name = r.split("==")[0]
    if name == "torch" and have_torch:
        continue
    mod = {"scikit-learn": "sklearn", "pyarrow": "pyarrow"}.get(name, name.replace("-", "_"))
    if importlib.util.find_spec(mod) is None:
        missing.append(r)
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
print(f"repo: {root}")


## 1. Data

Set the ticker and range. If Yahoo is unreachable from wherever you are running this, the cell falls back to a synthetic random walk so the rest of the notebook still executes. It says so loudly; do not mistake the fallback for a result.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.io as pio

pio.renderers.default = "colab" if "google.colab" in sys.modules else "notebook_connected"

from quantlab.config import DataConfig
from quantlab.data.loader import clean_ohlcv, integrity_report, load_ohlcv, synthetic_ohlcv

data_cfg = DataConfig(ticker="^OMXS30", start="2005-01-01", end="2025-12-31", interval="1d")

try:
    df = clean_ohlcv(load_ohlcv(data_cfg))
    SYNTHETIC = False
except Exception as e:  # network down, Yahoo sulking, proxy in the way
    print(f"Yahoo download failed ({e}).\nUsing SYNTHETIC data so the notebook runs. Nothing below means anything about {data_cfg.ticker}.")
    df = synthetic_ohlcv(3000, seed=42, start="2013-01-01")
    data_cfg = DataConfig(ticker="SYNTH", start="2013-01-01", end="2025-12-31")
    SYNTHETIC = True

report = integrity_report(df, data_cfg.interval)
print(pd.Series(report.summary()).to_string())
for w in report.warnings:
    print("WARNING:", w)


In [ ]:
from quantlab.ui import plots
plots.price_chart(df.tail(1500)).show()


## 2. Features

Every feature is shifted one bar before it enters the matrix. `build_feature_matrix` refuses `shift=0`, so you cannot accidentally hand the model today's close.

In [ ]:
from quantlab.config import FeatureConfig, FeatureSpec
from quantlab.features.registry import REGISTRY, _ensure_loaded, build_feature_matrix
_ensure_loaded()
print(f"{len(REGISTRY)} registered features:", ", ".join(sorted(REGISTRY)))

features = FeatureConfig([
    FeatureSpec("sma_ratio", {"window": 20}),
    FeatureSpec("ma_crossover", {"fast": 20, "slow": 50, "kind": "ema"}),
    FeatureSpec("macd", {}),
    FeatureSpec("roc", {"window": 10}),
    FeatureSpec("adx", {"window": 14}),
    FeatureSpec("rsi", {"window": 14}),
    FeatureSpec("bollinger", {"window": 20, "n_std": 2.0}),
    FeatureSpec("natr", {"window": 14}),
    FeatureSpec("realised_vol", {"window": 20}, transform="zscore", transform_window=250),
    FeatureSpec("vol_ratio", {"short": 10, "long": 60}),
    FeatureSpec("rolling_skew", {"window": 60}),
    FeatureSpec("drawdown", {}),
])
X_preview = build_feature_matrix(df, features.features)
print(X_preview.shape, "columns; first complete row:", X_preview.dropna().index[0].date())
plots.heatmap(X_preview.corr(), "Feature correlation").show()


## 3. Labels

Triple barrier with ATR-scaled barriers. Watch the class balance and the barrier breakdown: if almost everything hits the vertical barrier, the barriers are too wide and you are back to a fixed-horizon label with extra steps.

In [ ]:
from quantlab.config import LabelConfig
from quantlab.labeling import build_labels, class_balance

label_cfg = LabelConfig(scheme="triple_barrier", horizon=10, pt_mult=2.0, sl_mult=1.0, atr_window=14, weighting="uniqueness")
labels = build_labels(df, label_cfg)
bal = class_balance(labels["label"])
print(bal)
if bal.attrs["imbalanced"]:
    print(f"\nOne class is {bal.attrs['dominant_share']:.0%} of the sample. Accuracy is a garbage metric here.")
print("\nBarrier touched:\n", labels["barrier"].value_counts(normalize=True).round(3).to_string())
plots.labels_over_time(labels.tail(1000), df["close"].tail(1000)).show()


## 4. Model and walk-forward

Rolling windows, embargo equal to the label horizon, refit from scratch each fold. Set `retune_per_fold=True` for an Optuna study inside every fold (honest, slow).

In [ ]:
import time
from quantlab.config import ExperimentConfig, ModelConfig, WalkForwardConfig, BacktestConfig, CostConfig, SizingConfig
from quantlab.pipeline import prepare_dataset, run_walkforward

cfg = ExperimentConfig(
    seed=42,
    data=data_cfg,
    features=features,
    label=label_cfg,
    model=ModelConfig(family="lightgbm", params={"n_estimators": 300, "num_leaves": 15, "learning_rate": 0.03, "min_child_samples": 50}),
    walkforward=WalkForwardConfig(mode="rolling", train_window=1000, test_window=125, step=125, embargo=label_cfg.max_horizon, retune_per_fold=False, tune_iterations=10),
    backtest=BacktestConfig(
        long_threshold=0.55, short_threshold=0.55, direction="long_short", execution="next_open",
        costs=CostConfig(commission_bps=2.0, spread_bps=5.0, slippage_bps=2.0),
        sizing=SizingConfig(scheme="fixed_fractional", fixed_fraction=1.0),
    ),
)
cfg.validate()
print("config hash:", cfg.config_hash())

ds = prepare_dataset(df, cfg)
print(f"{ds.n} usable bars after trimming {ds.n_trimmed} warm-up rows, {len(ds.feature_columns)} feature columns")

def progress(ev: dict) -> None:
    if ev.get("stage") == "fold_done":
        print(f"  fold {ev['fold'] + 1:>2}/{ev['n_folds']}: test acc {ev['test_accuracy']:.3f}  ({ev['seconds']:.1f}s)")
    elif ev.get("stage") == "tune":
        print(f"    optuna trial {ev['candidate']}/{ev['n_candidates']}: val log-loss {ev['val_logloss']:.4f}")

t0 = time.time()
wf = run_walkforward(ds, cfg, progress)
print(f"\n{len(wf.folds)} folds, {len(wf.oos)} OOS bars, {time.time() - t0:.0f}s")
wf.fold_table[["fold", "train_start", "test_start", "test_end", "n_train", "n_test", "train_acc", "test_acc", "test_logloss"]]


## 5. Backtest and the honesty block

The DSR needs the number of things you have tried. It comes from the experiment log in `~/.quantlab/experiments/`, which this notebook appends to exactly like the app does.

In [ ]:
from quantlab.experiments.store import ExperimentStore
from quantlab.pipeline import evaluate, experiment_record

store = ExperimentStore()
is_new = (store.trials(cfg.data.ticker, cfg.data.interval)["config_hash"] == cfg.config_hash()).sum() == 0 if store.n_trials(cfg.data.ticker) else True
n_trials = store.n_trials(cfg.data.ticker, cfg.data.interval) + (1 if is_new else 0)

ev = evaluate(ds, wf, cfg, n_trials, store.trial_sharpes(cfg.data.ticker, cfg.data.interval), n_random=1000)
bt = ev.backtest

print("=== Is this real? ===")
print(f"Sharpe (OOS, net):      {ev.metrics['sharpe']:.2f}")
print(f"Deflated Sharpe:        {ev.dsr.dsr:.2f}   (trials counted: {ev.dsr.n_trials})")
print(f"Beats random strategies:{ev.random_bench.percentile:.0%}")
print()
print(ev.dsr.verdict())
print(ev.random_bench.verdict())

plots.random_benchmark_hist(ev.random_bench.random_sharpes, ev.random_bench.strategy_sharpe).show()


In [ ]:
plots.equity_curves(bt.equity, bt.benchmark_equity, folds=wf.fold_table).show()
m = pd.Series(ev.metrics)
m[["cagr", "ann_vol", "sharpe", "sortino", "calmar", "max_drawdown", "hit_rate", "profit_factor", "n_trades", "turnover_annual", "avg_exposure", "worst_month", "alpha_annual", "beta", "bench_sharpe", "bench_total_return", "total_return"]].round(4)


In [ ]:
plots.fold_bars(ev.fold_metrics).show()
print(f"{int((ev.fold_metrics['strategy_return'] > 0).sum())}/{len(ev.fold_metrics)} folds positive")
bt.trades.tail(10)


## 6. Diagnostics

Permutation importance is computed on OOS blocks with each fold's own model. SHAP explains the last fold.

In [ ]:
from quantlab.diagnostics import calibration, confusion, permutation_importance_oos, shap_last_fold, per_regime_performance
from quantlab.data.calendar import bars_per_year

pi = permutation_importance_oos(ds, wf, n_repeats=3, seed=cfg.seed, max_folds=5)
plots.bar(pi.set_index("feature")["importance"].iloc[::-1], "Permutation importance (OOS log-loss increase)", plots.GREEN).show()
noise = pi.loc[pi["importance"] < 0, "feature"].tolist()
if noise:
    print("Shuffling these improved OOS log-loss, i.e. noise the model latched on to:", ", ".join(noise))


In [ ]:
for blk in shap_last_fold(ds, wf, max_rows=400):
    print(blk.component, "on the", blk.scale, "scale")
    plots.shap_beeswarm(blk.values, blk.X, scale=blk.scale).show()


In [ ]:
print(confusion(wf))
plots.calibration_plot(calibration(wf)).show()
reg = per_regime_performance(ds, bt.returns, bt.benchmark_returns, bars_per_year(ds.interval))
plots.regime_bars(reg).show()


## 7. Log the run and compare

Logging is not optional in the app and it should not be optional here either: the trial count for the next DSR depends on it.

In [ ]:
from quantlab.experiments.compare import leaderboard, pbo_across_runs, parallel_coordinates_frame

if SYNTHETIC:
    print("Synthetic data: NOT logging, the experiment log is for real instruments only.")
else:
    rec = experiment_record(cfg, ds, wf, ev, session_id="notebook")
    try:
        run_id = store.log_run(rec, bt.returns)
        print("logged run", run_id, "->", store.runs_path)
    except ValueError as e:
        print("not logged:", e)


### Optional: a small sweep to populate the log

Runs a few model families through the identical geometry so the leaderboard and PBO have something to chew on. Each is logged. Comment out families you do not care about; the sequence models are the slow ones.

In [ ]:
SWEEP = {
    "logreg": {"C": 0.1, "l1_ratio": 0.5},
    "random_forest": {"n_estimators": 300, "max_depth": 5, "min_samples_leaf": 50},
    "xgboost": {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.03},
    # "gru": {"epochs": 30, "seq_len": 20, "hidden_size": 32},
}
import copy
sweep_results = {}
for fam, params in SWEEP.items():
    c = copy.deepcopy(cfg)
    c.model = ModelConfig(family=fam, params=params)
    d = prepare_dataset(df, c)
    w = run_walkforward(d, c)
    n = store.n_trials(c.data.ticker, c.data.interval) + 1
    e = evaluate(d, w, c, n, store.trial_sharpes(c.data.ticker, c.data.interval), n_random=300)
    sweep_results[fam] = e
    if not SYNTHETIC:
        try:
            store.log_run(experiment_record(c, d, w, e, session_id="notebook"), e.backtest.returns)
        except ValueError as err:
            print(fam, "not logged:", err)
    print(f"{fam:14s} sharpe {e.metrics['sharpe']:6.2f}  dsr {e.dsr.dsr:.2f}  trades {e.metrics['n_trades']:4d}  beats random {e.random_bench.percentile:.0%}")


In [ ]:
if not SYNTHETIC and store.n_trials(cfg.data.ticker, cfg.data.interval) >= 2:
    lb = leaderboard(store, cfg.data.ticker, cfg.data.interval)
    display(lb[["run_id", "model_family", "label_scheme", "m_sharpe", "dsr_now", "m_max_drawdown", "m_n_trades", "random_percentile", "n_trials_now"]].round(3))
    pbo = pbo_across_runs(store, lb["run_id"].tolist(), n_partitions=10)
    print(pbo.verdict())
    if np.isfinite(pbo.pbo):
        plots.pbo_hist(pbo.logits).show()
    plots.parallel_coords(parallel_coordinates_frame(lb)).show()
else:
    # Same machinery on the in-memory sweep so the cell still shows something on synthetic data.
    from quantlab.backtest.metrics import pbo_cscv
    M = pd.DataFrame({k: v.backtest.returns for k, v in {**sweep_results, cfg.model.family: ev}.items()})
    print(pbo_cscv(M, n_partitions=10).verdict())


## How to not fool yourself, the short version

- The **embargo** keeps overlapping labels out of the test window. It defaults to the label horizon and the config refuses less unless you say you know what you are doing.
- The **deflated Sharpe** discounts your best result by how many things you tried. The count comes from the log, so log everything.
- **PBO** across logged runs tells you how often the in-sample winner loses out of sample. Above 0.5 means your selection process is worse than a coin.
- The **random benchmark** shuffles your own trades. If you cannot beat 95% of them, you have a trading pattern, not an edge.

If all four look bad, the honest conclusion is that there is nothing here. That is a valid result and it is the usual one.